# 03 · Silver → Gold

Conceito praticado: **Star Schema**.
- Cinco dimensões, uma fato de desempenho e três tabelas-ponte.
- Grão da fato: **um filme lançado por linha**, com data conhecida e não futura.
- Chaves substitutas `BIGINT` geradas com `row_number()`.
- Contexto para IA agregado por filme.

As dimensões incluem o catálogo completo. Métricas ausentes permanecem nulas. As chaves podem mudar quando o catálogo muda: reexecute toda a Gold após atualizar a Silver.

In [ ]:
# Padronização de nomes: <catalog>.<camada>.<tabela>, tudo em snake_case
catalog = "cine_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"
landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/landing"

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

def validar_chave(df, colunas, nome):
    if df.groupBy(*colunas).count().filter(F.col("count") > 1).limit(1).count():
        raise ValueError(f"{nome}: chave duplicada em {colunas}")
    condicao = F.lit(False)
    for coluna in colunas:
        condicao = condicao | F.col(coluna).isNull()
    if df.filter(condicao).limit(1).count():
        raise ValueError(f"{nome}: chave nula")

def validar_fk(df, dimensao, chave, nome):
    if df.select(chave).join(dimensao.select(chave), chave, "left_anti").limit(1).count():
        raise ValueError(f"{nome}: referência sem dimensão em {chave}")

### Leitura da Silver e conferência das chaves

In [ ]:
df_info = spark.table(f"{silver_schema}.tb_info_filmes")
df_financeiro = spark.table(f"{silver_schema}.tb_financeiro_filmes")
df_metricas = spark.table(f"{silver_schema}.tb_metricas_engajamento")
df_generos = spark.table(f"{silver_schema}.tb_generos")
df_entidades = spark.table(f"{silver_schema}.tb_pessoas_empresas")
df_avaliacoes = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")

# Os joins da fato só podem usar fontes com uma linha por filme.
for nome, df in [("info", df_info), ("financeiro", df_financeiro), ("metricas", df_metricas)]:
    validar_chave(df, ["id_filme"], nome)

# Relações sem filme correspondente não podem gerar FKs órfãs.
# As contagens ficam visíveis; o conteúdo original permanece na Silver.
orfaos = []
for nome, df in [("financeiro", df_financeiro), ("metricas", df_metricas),
                 ("generos", df_generos), ("entidades", df_entidades), ("avaliacoes", df_avaliacoes)]:
    qtd = df.select("id_filme").distinct().join(df_info.select("id_filme"), "id_filme", "left_anti").count()
    orfaos.append((nome, qtd))
display(spark.createDataFrame(orfaos, "origem STRING, filmes_sem_cadastro BIGINT"))

## 1. dim_movies
Metadados dos filmes e chave substituta por `id_filme`.

In [ ]:
df_dim_movies = (
    df_info
    .withColumn("sk_movie_id", F.row_number().over(Window.orderBy("id_filme")).cast("bigint"))
    .select("sk_movie_id", "id_filme", "titulo", "data_lancamento", "ano_lancamento",
            "duracao_minutos", "idioma_original", "status_filme", "sinopse")
)
validar_chave(df_dim_movies, ["sk_movie_id"], "dim_movies")
df_dim_movies.write.format("delta").mode("overwrite") .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_movies")
# Usar as chaves gravadas nos relacionamentos seguintes.
df_dim_movies = spark.table(f"{gold_schema}.dim_movies")
df_chaves_filmes = df_dim_movies.select("id_filme", "sk_movie_id")
display(df_dim_movies.limit(10))

## 2. dim_genres
Catálogo único de gêneros.

In [ ]:
df_dim_genres = (
    df_generos.select("nome_genero").distinct()
    .withColumn("sk_genre_id", F.row_number().over(Window.orderBy("nome_genero")).cast("bigint"))
    .select("sk_genre_id", "nome_genero")
)
df_dim_genres.write.format("delta").mode("overwrite") .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_genres")
df_dim_genres = spark.table(f"{gold_schema}.dim_genres")
display(df_dim_genres.limit(10))

## 3. dim_people
Pessoas identificadas por nome e tipo de atuação: Ator, Diretor ou Roteirista.

In [ ]:
df_dim_people = (
    df_entidades.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .withColumnRenamed("nome_entidade", "nome_pessoa")
    .withColumnRenamed("tipo_entidade", "tipo_pessoa")
    .select("nome_pessoa", "tipo_pessoa").distinct()
    .withColumn("sk_person_id", F.row_number().over(Window.orderBy("nome_pessoa", "tipo_pessoa")).cast("bigint"))
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)
df_dim_people.write.format("delta").mode("overwrite") .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_people")
df_dim_people = spark.table(f"{gold_schema}.dim_people")
display(df_dim_people.limit(10))

## 4. dim_companies
Catálogo único de produtoras.

In [ ]:
df_dim_companies = (
    df_entidades.filter(F.col("tipo_entidade") == "Produtora")
    .withColumnRenamed("nome_entidade", "nome_produtora")
    .select("nome_produtora").distinct()
    .withColumn("sk_company_id", F.row_number().over(Window.orderBy("nome_produtora")).cast("bigint"))
    .select("sk_company_id", "nome_produtora")
)
df_dim_companies.write.format("delta").mode("overwrite") .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_companies")
df_dim_companies = spark.table(f"{gold_schema}.dim_companies")
display(df_dim_companies.limit(10))

## 5. dim_reviews
Uma linha por filme: quantidade de avaliações e média das notas válidas, arredondada em duas casas.

In [ ]:
# A contagem inclui avaliações com nota inválida; a média ignora notas nulas.
df_resumo_avaliacoes = df_avaliacoes.groupBy("id_filme").agg(
    F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
    F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios"),
)
df_dim_reviews = (
    df_chaves_filmes.join(df_resumo_avaliacoes, "id_filme", "left")
    .fillna({"qtd_avaliacoes_usuarios": 0})
    .withColumn("sk_review_id", F.row_number().over(Window.orderBy("sk_movie_id")).cast("bigint"))
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
)
validar_chave(df_dim_reviews, ["sk_movie_id"], "dim_reviews")
validar_fk(df_dim_reviews, df_dim_movies, "sk_movie_id", "dim_reviews")
df_dim_reviews.write.format("delta").mode("overwrite") .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_reviews")
display(df_dim_reviews.limit(10))

## 6. bridge_movie_genre
Relacionamento entre filmes e gêneros, sem duplicar o grão da fato.

In [ ]:
df_bridge_genre = (df_generos.join(df_chaves_filmes, "id_filme", "inner")
    .join(df_dim_genres, "nome_genero", "inner").select("sk_movie_id", "sk_genre_id").distinct())
validar_chave(df_bridge_genre, ["sk_movie_id", "sk_genre_id"], "bridge_movie_genre")
validar_fk(df_bridge_genre, df_dim_movies, "sk_movie_id", "bridge_movie_genre")
validar_fk(df_bridge_genre, df_dim_genres, "sk_genre_id", "bridge_movie_genre")
df_bridge_genre.write.format("delta").mode("overwrite") .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_genre")
display(df_bridge_genre.limit(10))

## 7. bridge_movie_person
Relacionamento entre filmes e pessoas, incluindo o tipo de atuação na dimensão.

In [ ]:
df_bridge_person = (
    df_entidades.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select("id_filme", F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa"))
    .join(df_chaves_filmes, "id_filme", "inner")
    .join(df_dim_people, ["nome_pessoa", "tipo_pessoa"], "inner")
    .select("sk_movie_id", "sk_person_id").distinct()
)
validar_chave(df_bridge_person, ["sk_movie_id", "sk_person_id"], "bridge_movie_person")
validar_fk(df_bridge_person, df_dim_movies, "sk_movie_id", "bridge_movie_person")
validar_fk(df_bridge_person, df_dim_people, "sk_person_id", "bridge_movie_person")
df_bridge_person.write.format("delta").mode("overwrite") .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_person")
display(df_bridge_person.limit(10))

## 8. bridge_movie_company
Relacionamento entre filmes e produtoras.

In [ ]:
df_bridge_company = (
    df_entidades.filter(F.col("tipo_entidade") == "Produtora")
    .select("id_filme", F.col("nome_entidade").alias("nome_produtora"))
    .join(df_chaves_filmes, "id_filme", "inner").join(df_dim_companies, "nome_produtora", "inner")
    .select("sk_movie_id", "sk_company_id").distinct()
)
validar_chave(df_bridge_company, ["sk_movie_id", "sk_company_id"], "bridge_movie_company")
validar_fk(df_bridge_company, df_dim_movies, "sk_movie_id", "bridge_movie_company")
validar_fk(df_bridge_company, df_dim_companies, "sk_company_id", "bridge_movie_company")
df_bridge_company.write.format("delta").mode("overwrite") .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_company")
display(df_bridge_company.limit(10))

## 9. fact_movies_performance
Consolidar métricas financeiras e de engajamento. As fontes dos joins têm uma linha por filme.

In [ ]:
df_lancados = df_dim_movies.filter(
    (F.col("status_filme") == "Lançado") & (F.col("data_lancamento") <= F.current_date())
)
colunas_financeiras = ["orcamento_usd", "receita_usd", "lucro_usd",
                      "orcamento_brl", "receita_brl", "lucro_brl"]
colunas_metricas = ["popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb"]
df_fact = (
    df_lancados.select("sk_movie_id", "id_filme")
    .join(df_financeiro.select("id_filme", *colunas_financeiras), "id_filme", "left")
    .join(df_metricas.select("id_filme", *colunas_metricas), "id_filme", "left")
    .select("sk_movie_id", *colunas_financeiras, *colunas_metricas)
)
for coluna in colunas_financeiras:
    df_fact = df_fact.withColumn(coluna, F.col(coluna).cast("decimal(18,2)"))
for coluna in ["qtd_votos_tmdb", "qtd_votos_imdb"]:
    df_fact = df_fact.withColumn(coluna, F.col(coluna).cast("int"))
for coluna in ["popularidade", "nota_media_tmdb", "nota_media_imdb"]:
    df_fact = df_fact.withColumn(coluna, F.col(coluna).cast("double"))
validar_chave(df_fact, ["sk_movie_id"], "fact_movies_performance")
validar_fk(df_fact, df_dim_movies, "sk_movie_id", "fact_movies_performance")
assert df_fact.count() == df_lancados.count(), "O join alterou o grão da fato."
df_fact.write.format("delta").mode("overwrite") .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.fact_movies_performance")
display(df_fact.limit(10))

## 10. tb_contexto_ia
O PDF menciona o contexto para IA, mas não define seu esquema. Usamos metadados, sinopse e listas de gêneros, pessoas e produtoras por filme. Cada lista é agregada antes do join para preservar o grão.

In [ ]:
df_contexto_generos = df_generos.groupBy("id_filme").agg(F.sort_array(F.collect_set("nome_genero")).alias("generos"))
df_contexto_entidades = df_entidades.groupBy("id_filme").agg(
    F.sort_array(F.collect_set(F.when(F.col("tipo_entidade") == "Ator", F.col("nome_entidade")))).alias("atores"),
    F.sort_array(F.collect_set(F.when(F.col("tipo_entidade") == "Diretor", F.col("nome_entidade")))).alias("diretores"),
    F.sort_array(F.collect_set(F.when(F.col("tipo_entidade") == "Roteirista", F.col("nome_entidade")))).alias("roteiristas"),
    F.sort_array(F.collect_set(F.when(F.col("tipo_entidade") == "Produtora", F.col("nome_entidade")))).alias("produtoras"),
)
df_contexto = (df_dim_movies.join(df_contexto_generos, "id_filme", "left")
               .join(df_contexto_entidades, "id_filme", "left"))
for coluna in ["generos", "atores", "diretores", "roteiristas", "produtoras"]:
    df_contexto = df_contexto.withColumn(coluna, F.coalesce(F.col(coluna), F.array().cast("array<string>")))
df_contexto = df_contexto.withColumn("contexto_textual", F.concat_ws("\n",
    F.concat(F.lit("Título: "), F.coalesce(F.col("titulo"), F.lit("Não Informado"))),
    F.concat(F.lit("Ano: "), F.coalesce(F.col("ano_lancamento").cast("string"), F.lit("Não Informado"))),
    F.concat(F.lit("Gêneros: "), F.concat_ws(", ", "generos")),
    F.concat(F.lit("Diretores: "), F.concat_ws(", ", "diretores")),
    F.concat(F.lit("Sinopse: "), F.coalesce(F.col("sinopse"), F.lit("Não Informada"))),
))
validar_chave(df_contexto, ["sk_movie_id"], "tb_contexto_ia")
df_contexto.write.format("delta").mode("overwrite") .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.tb_contexto_ia")
display(df_contexto.select("id_filme", "titulo", "generos", "diretores", "contexto_textual").limit(5))

## 11. Testando o Star Schema
Conferir chaves e contagens; as seis perguntas de negócio estão no notebook 04.

In [ ]:
resumo = []
for tabela, chave in [("dim_movies", "sk_movie_id"), ("dim_genres", "sk_genre_id"),
                      ("dim_people", "sk_person_id"), ("dim_companies", "sk_company_id"),
                      ("dim_reviews", "sk_review_id"), ("fact_movies_performance", "sk_movie_id")]:
    df = spark.table(f"{gold_schema}.{tabela}")
    validar_chave(df, [chave], tabela)
    resumo.append((tabela, df.count(), df.select(chave).distinct().count()))
display(spark.createDataFrame(resumo, "tabela STRING, linhas BIGINT, chaves_distintas BIGINT"))

## Recapitulando a arquitetura construída

| Camada | Conteúdo |
| --- | --- |
| Landing | Cinco CSVs em um Volume |
| Bronze | Ingestão dos CSVs e da cotação em Delta/Append |
| Silver | Limpeza, tipagem, deduplicação e log de qualidade |
| Gold | Fato, cinco dimensões, três pontes e contexto para IA |